# Atlas IoT Application IDE: Codebase Study Guide

> Covers `Final_Project/` of `alessandroquerze/boar_iot` at commit `0902eb0` ("Drag relationships onto the editor").
> Everything below was checked against the source. The test suite was run: **134 / 134 pass** on Node 22.
> File paths are relative to `Final_Project/` unless stated otherwise.

---

## Table of contents

1. [The 60-second pitch](#1-the-60-second-pitch)
2. [The assignment: what the project has to do](#2-the-assignment-what-the-project-has-to-do)
3. [Architecture and data flow](#3-architecture-and-data-flow)
4. [Repository map](#4-repository-map)
5. [Build, run, test](#5-build-run-test)
6. [The Atlas middleware (C++, on the Raspberry Pi)](#6-the-atlas-middleware-c-on-the-raspberry-pi)
7. [Tweets and the wire protocol](#7-tweets-and-the-wire-protocol)
8. [Server: `server/index.js`, HTTP API and SSE](#8-server-serverindexjs-http-api-and-sse)
9. [Server: the space model (`server/space.js`)](#9-server-the-space-model-serverspacejs)
10. [Server: the gateways (real Atlas and simulator)](#10-server-the-gateways-real-atlas-and-simulator)
11. [Server: apps, relationships, runtime](#11-server-apps-relationships-runtime)
12. [The composition language (`lang/`)](#12-the-composition-language-lang)
13. [The orchestrator (`lang/run.js`)](#13-the-orchestrator-langrunjs)
14. [The graph model (`lang/graph.js`)](#14-the-graph-model-langgraphjs)
15. [The browser front end (`public/`)](#15-the-browser-front-end-public)
16. [Keeping the two editors in sync (the core mechanism)](#16-keeping-the-two-editors-in-sync-the-core-mechanism)
17. [Relationships, end to end](#17-relationships-end-to-end)
18. [Walkthroughs: what happens when…](#18-walkthroughs-what-happens-when)
19. [Tests](#19-tests)
20. [Design decisions and documented conflicts](#20-design-decisions-and-documented-conflicts)
21. [Known limitations, gotchas and doc-vs-code differences](#21-known-limitations-gotchas-and-doc-vs-code-differences)
22. [Likely questions and answers](#22-likely-questions-and-answers)
23. [Glossary](#23-glossary)

---

## 1. The 60-second pitch

The project is a **VS Code-style web IDE for building IoT apps on top of the Atlas IoT middleware**. It has two halves:

- **On the Raspberry Pi**, the Atlas middleware (C++) reads an XML description of the thing (the IoT-DDL), compiles each service into a plugin (`.so`), announces the thing over **MQTT** as JSON "tweets", and executes **service calls** that arrive over **TCP port 6668**.
- **On the developer's PC**, a small **Node.js server** listens to those tweets and builds a live model of the smart space. It stores apps as `.iot` files and runs them by sending TCP service calls. A **browser UI** shows a **Drawflow** graphical editor and a **CodeMirror 6** text editor, kept in sync through one shared AST.

Apps are written in a tiny language inherited from Lab 4: `RUN … THEN …` (order-based), `IF … DO … ELSE … END` (condition-based), plain sequences, and `LOOP EVERY Ns … END`. The success criterion is to rebuild the Lab 4 Part II demo app (temperature → fan + window) more easily than before.

Key numbers: browser ↔ server on **HTTP :7000** (REST + Server-Sent Events). Server ↔ things on **MQTT :1883** for tweets and **TCP :6668** for calls. Apps live in `apps/*.iot`. User-defined relationships live in `apps/relationships.json`.

---

## 2. The assignment: what the project has to do

Two sources of requirements exist, and **the task prompt wins over the course guidelines** when they conflict (conflicts are logged in `directive_conflicts.md`).

### 2.1 Course guidelines (`docs/Guidelines.pdf`, summarised in `PLAN.md` §2)

| Req | Meaning | Where it is met |
|---|---|---|
| **R1 Discover** | A background listener learns the space from five tweet types: thing identity, entity identity, service, thing language, relationship. Freeze the UI at start-up until discovery settles. | `server/gateway-atlas.js` + `server/space.js`; overlay in `public/main.js` |
| **R2 Display** | Show the IDE's network address. Show things, services (alphabetical, sensor/actuator split), relationships and apps. | Meta bar + Devices panel + Apps panel |
| **R3 Editor** | Compose Type 1 (sequence), Type 2 (order-based), Type 3 (condition-based) apps, guided by relationships. Offer Clear and Finalize. | `lang/` + graph and text editors |
| **R4 Manager** | SAVE (`.iot`, naming rules, working directory), UPLOAD (double-click), ACTIVATE (status panel row with start time), STOP (cleanup; finished apps show *Completed* and cannot be stopped), DELETE (Apps tab only) | `server/apps.js`, `server/runtime.js`, `public/panel-apps.js` |

Deliverables: zip + report (≤5 pages, with **open-source justification** and **screenshots of dialogs**), a 2-minute deck and a live demo. Golden rule: **ship only features that work**.

### 2.2 Task prompt (verbatim in `PLAN.md` §1)

- VS Code layout. The menu bar holds `activate`, `update` (renamed **`save`** later), `stop` and `delete`. The top bar shows the network address. The activity bar has exactly two icons, **Apps** and **Devices**. The main area holds the **graphical editor (Drawflow)**, the right pane the **text editor (CodeMirror 6)**, and the bottom panel the **console**. There is one tab per open project.
- **Graph → text**: added block/connection → its lines turn **green**. Removed → its lines turn **red**, then the empty lines are removed automatically.
- **Text → graph**: an **Apply Changes** button. The graph is **locked and blurred** from the first keystroke until Apply or Discard. Validation must be lightweight lexical/syntactic only (reuse Lezer, no compiler). On failure: red highlights, errors in the console, nothing applied.
- **Settings**: highlighting can be turned off.
- **Multiple projects**: each one fully independent.

### 2.3 Instructor update (27 Nov): user-defined relationships

The IoT-DDL Builder can only relate services **of the same entity**, and Atlas advertises those relationships but never executes them. So the IDE must keep listening for advertised relationships and **also** let the developer add `order` / `condition` relationships between **any two services**, across entities and across things. This is implemented (see [§17](#17-relationships-end-to-end)). The project doc `claude/ROADMAP-user-relationships.md` is the plan for it.

---

## 3. Architecture and data flow

### 3.1 Why a server at all

A browser alone cannot (1) open raw TCP sockets to port 6668, (2) scan, write and delete files in a working directory, or (3) keep an app running after the tab closes. So Node.js owns **discovery, files and execution**, and the browser owns **the IDE**. This is decision **D1**.

### 3.2 The picture

```
 Browser (the IDE)                             Node.js server  (server/index.js, port 7000)
 ┌─────────────────────────────────┐  fetch    ┌──────────────────────────────────────────┐
 │ main.js       shell, tabs, menu │ ────────► │ REST routes  /api/state /api/apps …      │
 │ graph-editor  Drawflow canvas   │           │ space.js     the space model (R1/R2)     │
 │ text-editor   CodeMirror 6      │ ◄──────── │ apps.js      .iot files (R4)             │
 │ panel-*       apps / devices    │   SSE     │ relationships.js  relationships.json     │
 │ bottom-panel  console/out/status│ /api/events│ runtime.js  activate / stop (R4)        │
 │ editor.bundle = CodeMirror+lang/│           │ lang/        parse + run (shared code)   │
 └─────────────────────────────────┘           └────────┬───────────────────────┬─────────┘
                                                        │ MQTT :1883            │ TCP :6668
                                          (subscribe /Atlas/#)          (one JSON call per connection)
                                                        ▼                       ▼
                                          ┌──────────────────┐   ┌──────────────────────────────┐
                                          │ Mosquitto broker │◄──│ Atlas middleware on the RPi  │
                                          └──────────────────┘   │ (publishes tweets every 10 s)│
                                                                 └──────────────────────────────┘
      or, with --simulate: server/gateway-sim.js replaces both arrows (no hardware)
```

### 3.3 Stack and third-party libraries (all MIT, all vendored, no CDN)

| Library | Where | Why |
|---|---|---|
| **Drawflow** 0.0.60 | `public/vendor/drawflow.min.*` (copied by `build.mjs`) | Required by the prompt. Drag-and-drop nodes with typed inputs/outputs, `export()`/`import()` as the bridge to the AST |
| **CodeMirror 6** (`state`, `view`, `language`, `commands`, `lint`) | bundled into `public/vendor/editor.bundle.js` | Required by the prompt. Decorations for green/red, transactions with annotations to tell "our" edits from the user's |
| **Lezer** (`lr`, `highlight`; `generator` at build time) | `lang/iot.grammar` → `lang/iot.parser.js` | One declarative grammar gives tokens, highlighting and **error nodes with exact positions** (the lexical and syntactic validation) |
| **MQTT.js** (`mqtt`) | server only | The things publish tweets over MQTT. This is the only runtime dependency |
| **esbuild** | build only | Bundles CodeMirror + `lang/` into one IIFE exposed as `window.AtlasEditor` |

Node built-ins only on the server (`http`, `net`, `fs`, `crypto`). **No web framework.** Server → browser uses **SSE** rather than WebSockets because the stream is one-way and SSE needs no dependency. The UI is plain ES modules with no framework.

On the Pi (C++): **CppMicroServices** (service plugins), **Paho MQTT C**, **RapidJSON**, **tinyxml2**, **wiringPi/pigpio**, **GNU AutoGen** (code generation), plus legacy Wakaama (OMA LwM2M) and CoAP libraries that are compiled in but unused.

---

## 4. Repository map

```
Final_Project/
├─ README.md            original group planning notes + run instructions + licence table
├─ PLAN.md              THE plan (requirements verbatim, architecture, file map, steps 0–12, decisions D1–D13)
├─ PROJECT.md           1-line pointer to PLAN.md (AGENTS.md naming)
├─ AGENTS.md            rules for AI coding sessions (readability, commits, checkpoints)
├─ CHECKPOINT.md        where the work stands (steps 0–10 done; step 11 hardware pass blocked)
├─ QUESTIONS.md         closed; answers became D1–D13
├─ directive_conflicts.md   conflicts C1–C9 (prompt vs guidelines)
├─ package.json / build.mjs
├─ apps/                default Atlas working directory: AirControl.iot, NightWatch.iot
├─ docs/  tweets.md (tweet shapes), Guidelines.pdf, guideline-conflicts.md (pointer)
├─ lang/                the language, shared by browser and server
│   iot.grammar  →  iot.parser.js + iot.terms.js (generated, committed)
│   ast.js  print.js  validate.js  graph.js  run.js
├─ server/  index.js space.js gateway-atlas.js gateway-sim.js apps.js relationships.js runtime.js
├─ public/  index.html style.css main.js project.js text-editor.js graph-editor.js
│           editor-entry.js panel-apps.js panel-devices.js relationship-guidance.js
│           bottom-panel.js settings.js resizers.js  vendor/  dev/probe.html
├─ tests/   lang graph run space apps relationships server (*.test.js)
└─ Atlas/   the C++ middleware to copy onto the Raspberry Pi
    README.md (Italian deploy guide), build.sh, run.sh, bin/Atlas (prebuilt ARM64)
    Main/AtlasMain.cpp + CMakeLists.txt
    Architecture/*.cpp|h   (Communication_Manager, API_Generator, Identity_Parser,
                            Knowledge_Tweeting_Manager, IoTDDL_Parser, + unused modules)
    Architecture/GeneratedServices/  service.tpl, Service.cpp (interface), Makefile
    ConfigurationFiles/Atlas_IoTDDL.xml   the thing's description (source of truth)
    lib/   third-party C/C++ libraries (excluded from the project sync)
```

Size: about 6,700 lines of hand-written JS (server + lang + public + tests, not counting the generated parser or vendored bundles) and about 5,900 lines of C++ in `Atlas/Architecture`.

Outside `Final_Project/` the repo holds `Lab1`–`Lab4`. **Lab 4** is the reference: its mini-IDE (`Lab4/PartOne/edge/app.js`) provided the language and orchestrator, and `Lab4/PartTwo/atlas_demo.cpp` defined the TCP call format.

---

## 5. Build, run, test

```bash
cd Final_Project
npm install
npm run build                 # grammar -> lang/iot.parser.js ; CodeMirror+lang -> public/vendor/editor.bundle.js ; copy Drawflow
node --test "tests/*.test.js" # 134 tests. (`npm test` = `node --test tests/` FAILS on Node 22: directory arg)
npm start -- --simulate       # no hardware: two simulated things
npm start                     # real space, default broker 192.168.8.189:1883
node server/index.js --broker 25.16.192.196:1883 --apps /some/dir --port 7000
```

**Server flags** (`parseArgs` in `server/index.js`): `--port` (default 7000), `--simulate`, `--broker host:port` (default `192.168.8.189:1883`), `--apps dir` (default `Final_Project/apps`, resolved against the current directory). Any other flag throws `unknown option`.

**URL query parameters** (handled in `public/main.js`):

| Param | Effect |
|---|---|
| `?panel=devices` | open a side panel directly |
| `?sample=1` | open the Lab 4 Part II demo app as an unsaved project |
| `?new=1` | open an empty project |
| `?live=0` | poll `/api/state` + `/api/runs` every 400 ms instead of SSE (for the headless probe) |

**Rebuild rule**: after any change to `lang/` or `public/editor-entry.js`, run `npm run build` and **commit the outputs** (`lang/iot.parser.js`, `lang/iot.terms.js`, `public/vendor/editor.bundle.js`), because the app loads them directly.

**On the Raspberry Pi** (`Atlas/README.md`, in Italian): copy `Atlas/` to the Pi, then `sudo sh run.sh` (it uses `bin/Atlas` if present, else `build/Atlas`; `--build` forces `build/Atlas`). Rebuild with `sh build.sh` (`cmake -S Main -B build`). Both scripts refuse to run unless the Pi is **64-bit ARM (aarch64)**. `run.sh` adds `lib/bin` to `PATH` (for `usResourceCompiler4`) and `lib/lib` to `LD_LIBRARY_PATH`. Verify with `ss -ltn 'sport = :6668'` and `mosquitto_sub -h <broker> -t '/Atlas/#' -v`.

> ⚠️ The current IoT-DDL points at a **Hamachi VPN** setup (broker `25.16.192.196`, interface `ham0`), but the server's default broker is `192.168.8.189`. With the current XML you must pass `--broker 25.16.192.196:1883`.

---

## 6. The Atlas middleware (C++, on the Raspberry Pi)

### 6.1 Start-up sequence (`Main/AtlasMain.cpp` → `Communication_Manager`)

```
main()
 └─ Communication_Manager manager;          // constructor does all the preparation:
     1. IoTDDL_Parser reads ConfigurationFiles/Atlas_IoTDDL.xml → Thing_ATID, Thing_SSID (throws if empty)
     2. GN.Generate_ServicesBundles()        // compile every service into lib<Name>.so  (see 6.3)
     3. Build the tweets (all kept as JSON strings):
          IDP.parse_ThingIdentity()   → Identity_Thing
          IDP.parse_ThingLanguages()  → Identity_Language (IP read from the interface named in <Module>, e.g. ham0; Port "6668")
          IDP.parse_EntityIdentity()  → one Identity_Entity per entity
          KTM.parse_ThingServices()   → one Service tweet per service (with the API string)
          KTM.parse_ThingRelationships() → one Relationship tweet per IoT-DDL relationship
     4. Read MQTT broker IP/port and topic names from the XML; validate them; QoS = 1
 ├─ std::thread publisher(MQTT_Publish)     // forever: connect, publish ALL tweets, sleep 10 s, repeat
 └─ manager.CaptureAPIcalls_IPSocket()      // main thread: TCP server on :6668, one call per connection
```

`MQTT_Publish` publishes on `RootTOPIC + suffix`: identity and language tweets go to `/Atlas/Multicast/Tweet_ThingIdentity`, entity tweets to `/Atlas/Multicast/Tweet_EntityIdentity`, and service and relationship tweets to `/Atlas/Multicast/API`. On a publish failure it disconnects, waits 3 s and reconnects. Republishing the whole catalogue every 10 s is what lets an IDE started later still discover the thing.

`CaptureAPIcalls_IPSocket`: `accept` → a single `read` of up to 4048 bytes → RapidJSON parse → if `Thing ID` ≠ own ATID reply `{ "Status" : "ATID doesn't match" }`, if `Space ID` differs reply `SSID doesn't match`, else `GN.Handle_ServiceCall(json)` → `write` the reply → `close`. The old Lab 4 `wait(3)` pause after each call is **commented out**, so the next call is accepted as soon as the previous socket closes.

### 6.2 `IoTDDL_Parser`

A wrapper around **tinyxml2**. `parseXMLTag(tag1, …, tagN)` (4–8 levels) walks `FirstChildElement` down the path and returns the text, or `""` if any level is missing (helper `readTagText`). The file path is fixed: `ConfigurationFiles/Atlas_IoTDDL.xml`, **relative to the current directory**. That is why `run.sh` does `cd` to the `Atlas` folder first. The file also has `retriveXMLTag`, `updateXMLTag` and `addXMLTag` (legacy).

### 6.3 Turning XML services into plugins (`API_Generator::Generate_ServicesBundles`)

For each `Entity_j/Services/Service_i`:

1. Read `Name`, `Service_NumberInputs`, `Service_InputTypes`, `Service_InputDescriptions`, `Libraries`, `ADC_Number`, `OutputType`, `OutputDescription` and **`Service_Formula`** (C code in a CDATA block).
2. Write `Architecture/GeneratedServices/service/<Name>/service.def`, an **AutoGen** definitions file with `library = "#include <…>"`, `output = { name; type }`, `inputs = { name; type }`, `name = …`, and one `source = "…";` per formula line.
3. Translate the formula line by line. Lines containing `//`, `return`, `cout`, `![` or `]]` are **skipped**. Macros `DigitalWrite_High(p)`, `DigitalWrite_Low(p)`, `Delay(ms)`, `DigitalWrite_Blink(p,ms)` and analog reads expand to wiringPi calls. Everything else is copied as-is. The optional `ADC_Number` adds ADS1115 (I²C) or MCP3008 (SPI) reading code.
4. `make -C Architecture/GeneratedServices/service <Name>` runs `autogen -T service.tpl` → `main.cpp` + `manifest.json`, then `g++` → `main.o`, then `g++ -shared` → `lib<Name>.so`, then `usResourceCompiler4` embeds the manifest zip.

The template (`ServiceTemplate/service.tpl`) generates a class `Service<Name> : public Service` with:
- the output variable as a member (named after `OutputDescription`, e.g. `Fan_speed`),
- `void* Evaluate(va_list)`: reads each input with `va_arg`, runs the formula, returns `&output` (or `NULL` for void),
- `bool Validate(types, returnType)`: checks argument count and `typeid` of every argument and the return,
- a `BundleActivator` that registers the service with CppMicroServices.

**Consequence worth knowing:** a formula cannot use `return`, because those lines are skipped. It **assigns the output variable instead** (`Fan_speed = speed;`). It also cannot contain `//` comments.

### 6.4 Executing a call (`API_Generator::Handle_ServiceCall`)

1. Create a **new CppMicroServices framework on every call**, `Init()`, get the context.
2. Re-check Thing ID / Space ID. On mismatch: `Status: Unsuccessful`, `Status Description: incorrect thing id or smart space id`.
3. Loop over the IoT-DDL services to find `Service Name`. Then `InstallBundles(".../<Name>/lib<Name>.so")` and start the framework and its bundles.
4. Parse `Service Inputs`: `()`, `(null)`, `(NULL)`, `(void)` and `(VOID)` mean none. Otherwise strip the parentheses and split on `,` into **integers** (`stol`).
5. Dispatch on the shape. Only **0, 1 or 2 integer inputs** and **`void` or `int` output** are supported. It calls `invoke<int(int)>(ctx, name)(x)` and so on. The `invoke<Out(Ins...)>` template finds the service by bundle symbolic name, calls `Validate` with the C++ `typeid`s, and then `Evaluate` through a `va_list`.
6. Reply, as hand-built JSON:
   ```json
   { "Tweet Type" : "Service call reply", "Thing ID" : "…", "Space ID" : "…", "Service Name" : "Fan_On",
     "Status" : "Successful", "Status Description" : "Successful Service Call", "Service Result" : "60" }
   ```
   `"Service Result" : "No Output"` for void. Failures carry `Status: Unsuccessful` and `Status Description` of `Unknown Service` or `Incorrect service arguments`.

### 6.5 The IoT-DDL of this group (`ConfigurationFiles/Atlas_IoTDDL.xml`)

- Thing **`AT_BoarAirSensor`** ("Smart Air Quality Sensor", owner BoarGroup, OS Raspbian), space **`BoarSmartSpace`**.
- Network: `Module=ham0`, `Network_Name=Hamachi`, `Communication_Protocol=MQTT`, broker `25.16.192.196:1883`, topic root `/Atlas`.
- **3 entities, 10 services, 6 relationships:**

| Entity | Service | Type | Input | Output |
|---|---|---|---|---|
| `ClimateSensor` (DHT11) | `Calibrate_Sensor` | Report | – | `Calibration:int` (1 = sensor read OK, −1 = failed) |
| | `Read_Temp` | Report | `F_or_C` (0 = °C, 1 = °F) | `Temperature:int` (−1 if the DHT11 fails 3 tries) |
| | `Read_Humidity` | Report | – | `Humidity:int` |
| `Fan` (motor + 2N2222) | `Fan_On` | Action | `speed` (clamped 0–75) | `Fan_speed:int` |
| | `Fan_Off` | Action | – | void |
| | `Read_Fan_Speed` | Report | – | `Fan_speed:int` |
| | `Read_Fan_Status` | Report | – | `Fan_status:int` (1 if speed > 0) |
| `Window` (servo) | `Set_Window` | Action | `angle` (clamped 0–50) | `Window_angle:int` |
| | `Window_Close` | Action | – | void |
| | `Read_Window_Position` | Report | – | `Window_angle:int` |

Relationships: `order` *Calibrate before temperature* (`Calibrate_Sensor → Read_Temp`) and *Calibrate before humidity* (`→ Read_Humidity`) on ClimateSensor. Four `interfere` ones: `Fan_On↔Fan_Off` and `Set_Window↔Window_Close`, each in both directions. **There is no `condition` relationship in the XML.**

Hardware details from the formulas (BCM pin numbering via `wiringPiSetupGpio`): DHT11 on **GPIO 17** with 3 attempts 1.1 s apart and a checksum check. Fan PWM on **GPIO 13**, servo PWM on **GPIO 18**. PWM clock 192, range 2000 → 19.2 MHz/192/2000 = **50 Hz**. Fan duty = `speed*2000/100`. Servo = `150 + angle*200/180` ticks. Actuators cannot be read back, so `Fan_On`/`Set_Window` write the commanded value to **`/tmp/fan_speed`** and **`/tmp/window_angle`**, and the `Read_*` services read those files.

### 6.6 What is compiled in but not used

`Attachment_Engine` (only constructed as a global), `Objects_Engine` (OMA LwM2M device management), `System_Functions` (sample functions from the original Atlas), the UDP multicast functions `BroadcastTweets_IPSocket` (232.1.1.1:1235) and `CaptureMatesTweets_IPSocket`, `MQTT_Subscribe` / `msgarrvd` (listening to other things), `InitiateAPICall_IPSocket`, the Knowledge_Tweeting_Manager service graph, "unbounded services" and the commented-out CryptoPP encryption, and the `ConsumerTemplate`. This version of Atlas only **announces** and **serves**. It does not consume other things.

---

## 7. Tweets and the wire protocol

All tweets are **flat JSON objects whose keys contain spaces**. They are hand-built with spaces around the colons, so they must be parsed, never pattern-matched. Every tweet has `Tweet Type`, `Thing ID` and `Space ID`. Full reference with source line numbers: `docs/tweets.md`.

| Tweet Type | Main fields | IDE uses it for |
|---|---|---|
| `Identity_Thing` | `Name, Model, Vendor, Owner, Description, OS` | the thing card |
| `Identity_Language` | `Network Name, Communication Language, IP, Port` ("6668") | **where to send calls** (`thing.ip`, `thing.port`) |
| `Identity_Entity` | `Name` (label), `ID` (entity id), `Type, Owner, Vendor, Description` | the entity groups |
| `Service` | `Name, Entity ID, Type` (`Report`/`Action`), `API`, `Keywords, Description, …` | the service palette, sensor/actuator split, the catalogue |
| `Relationship` | `Name, Type, Description, FS name, SS name, Category` (always "Cooperative"), `Owner` | the advertised relationships |

**The `API` string** packs the signature: `<name>:[<inputs>]:(<output>)`. Each input is `desc,type, NULL` (note the space before `NULL`, an unused range slot), inputs are joined with `|`, and a lone `NULL` means none.
- `Read_Temp:[F_or_C,int, NULL]:(Temperature,int, NULL)`
- `Fan_Off:[NULL]:(NULL)`

It is parsed by `parseServiceApi()` in `server/space.js`.

**Service call** (IDE → thing, TCP 6668, one connection per call, the thing closes after replying):
```json
{ "Tweet Type": "Service call", "Thing ID": "AT_BoarAirSensor", "Space ID": "BoarSmartSpace",
  "Service Name": "Fan_On", "Service Inputs": "(60)" }
```
`Service Inputs` is a parenthesised comma list, `"()"` if empty. `Status` is `Successful` / `Unsuccessful`. A failed DHT11 read returns **−1 with `Status: Successful`**, because Atlas has no error status for a service body.

> Important caveat (CHECKPOINT "Known issues"): these shapes were **read from the middleware source**, not captured live. `gateway-atlas.js` had not yet been run against a real thing. Plan step 11 (the hardware pass) is still open.

---

## 8. Server: `server/index.js`, HTTP API and SSE

The HTTP entry point. It wires every piece together:

```js
space         = new Space({ id: "BoarSmartSpace", ideAddress: `http://${lanAddress()}:${port}` })
gateway       = simulate ? createSimGateway({onTweet}) : createAtlasGateway({broker, onTweet, onStatus})
apps          = new AppStore(appsDir)
relationships = new RelationshipStore(appsDir)
runs          = new RunManager({ gateway, space, onChange: → broadcast("runs"), onLog: → broadcast("log") })
```

`lanAddress()` returns the first non-internal IPv4 address (R2 "network address where the IDE is launched"). If there is none, it returns `localhost`.

### 8.1 REST routes

| Method + path | Answer |
|---|---|
| `GET /api/state` | `space.toJSON()` (see §9.3) |
| `GET /api/events` | **SSE stream**. Sends `state` and `runs` immediately, then pushes `state`, `apps`, `runs` and `log` |
| `GET /api/config` | `{ appsDir, simulate, broker }` |
| `GET /api/apps` | list of `{ file, name, modified }`, newest first |
| `GET /api/apps/:file` | the parsed `.iot` + `file` |
| `PUT /api/apps/:file` | body `{source, layout}` → save. `400 {error}` if `nameError`. Broadcasts `apps` |
| `DELETE /api/apps/:file` | unlink. Broadcasts `apps`. Returns `{deleted}` |
| `POST /api/apps/:file/activate` | reads the file **from disk** and starts a run → `{id}`. `409` if already running |
| `GET /api/runs` | the run list |
| `POST /api/runs/:id/stop` | `200` + list, or `409 {error}` (unknown run, or not Active) |
| `GET /api/relationships` | `space.toJSON().relationships` |
| `POST /api/relationships` | `201` + record, or `400 {error}`. Then `space.setUserRelationships()` → `state` broadcast |
| `DELETE /api/relationships/:id` | `204`. `403` for an advertised one, `404` for an unknown one |
| `GET` anything else | static file from `public/` (path-traversal-safe, `cache-control: no-store`) |

Errors: `ENOENT` → 404, anything else → 500 `{error}`. Request bodies are capped at 1 MB.

### 8.2 Start-up order (in the `listen` callback, which is deliberate)

1. `apps.ready()` (mkdir the working directory)
2. `relationships.load()` → `space.userRelationships = saved`, so the first state already has them
3. Print `Atlas IoT IDE on http://…` (the tests wait for this first line)
4. Subscribe `space.on("change") → broadcast("state")`. With `--simulate`, set broker = `simulated`
5. `space.startScanning()` then `gateway.start()`, so the scan begins only once the IDE is reachable

`SIGINT`/`SIGTERM` → `runs.stopAll()`, `gateway.stop()`, `space.stop()`, close, exit. Runs are not persisted, so the status panel is empty on every start (R4).

---

## 9. Server: the space model (`server/space.js`)

`class Space extends EventEmitter`. It is the single source of truth for "what exists in the smart space". It emits `change` with the full JSON on every update.

### 9.1 Merging tweets: `applyTweet(tweet)`

- Ignored unless it has both `Tweet Type` and `Thing ID`. Unknown types return `false` (not fatal).
- Updates `lastTweetAt` (for the scan) and `thing.lastSeen` (for staleness).
- `thingFor(id)` / `entityFor(thing, id)` **create on first sight**, so tweet order does not matter. A Service tweet arriving before its entity creates that entity.
- **Merge keys** (repeats update, never duplicate): thing by `Thing ID`, entity by `ID` (or `Name` if no ID), service by `(entity, Name)`, relationship by `name|first|second`.
- Services are **sorted alphabetically** (`localeCompare`) after every insert (R2). Entities are sorted by id.
- Relationship `Type` is lower-cased.
- If the space id is still unknown, it adopts the tweet's `Space ID`.

### 9.2 The start-up freeze

| Constant | Value | Meaning |
|---|---|---|
| `QUIET_MS` | 3000 | discovery is `ready` once no tweet has arrived for 3 s (and at least one has) |
| `MAX_SCAN_MS` | 15000 | …or after 15 s regardless, so the IDE never stays frozen |
| `STALE_MS` | 120000 | a thing silent for 2 min is flagged `stale` (shown, not removed) |

A 500 ms `setInterval` runs `checkScanFinished()`. `discovery: "scanning" | "ready"` drives the blocking overlay in the browser. The real Atlas republishes every 10 s, so the 3 s quiet window closes after the first burst. The simulator emits 27 tweets 60 ms apart (≈1.6 s), so "ready" arrives about 4.6 s after start.

### 9.3 What the browser gets: `toJSON()`

```jsonc
{ "space": { "id", "ideAddress", "broker": "connected|connecting|disconnected|error|simulated", "discovery": "scanning|ready" },
  "things": [{ "id", "spaceId", "name", "owner", "description", "os", "model", "vendor", "ip", "port", "network", "language",
               "lastSeen", "stale",
               "entities": [{ "id", "name", "type", "description", "role": "sensor|actuator|unknown",
                              "services": [{ "name", "type": "Report|Action", "inputs": [{name,type}], "output": {name,type}|null,
                                             "keywords": [], "description" }] }],
               "relationships": [{ "name", "type", "description", "first": "Fan_On", "second": "Fan_Off" }] }],
  "relationships": [ /* advertised + user, fully qualified, see §17 */ ] }
```

`roleOf(entity)`: no services → `unknown`; **any** `Action` service → `actuator`; otherwise `sensor` (decision D5). So the Fan, which has `Read_Fan_Speed` (Report) too, counts as an actuator.

### 9.4 Relationship resolution: `relationshipList()` and `endpoint()`

Relationship tweets carry only bare service names, and **can arrive before the Service tweet**. So the entity of each end is resolved **at serialisation time**, not when the tweet arrives. `endpoint(thing, service, fallbackEntity)` searches that thing's entities and returns `{thing, entity, service, found}`. `resolved = first.found && second.found`. Advertised entries come first and get the id `iot-ddl|<thing>|<name>|<first>|<second>`, which stays stable across re-announcements. User entries are re-resolved every time, keeping their stored entity as a fallback when the thing is offline.

---

## 10. Server: the gateways (real Atlas and simulator)

Both implement the same tiny interface, so nothing else knows which one is active:

```
{ name, start(), stop(), call({thing, space, service, args, ip, port}) → Promise<{status, result, description}> }
```

### 10.1 `gateway-atlas.js` (real)

- **Discovery**: `mqtt.connect("mqtt://host:port", {reconnectPeriod: 4000})`. On connect it subscribes to **`/Atlas/#`** (QoS 0). Each message is JSON-parsed and passed to `onTweet`. It routes on the `Tweet Type` field, **not on the topic**, because the topic split was the least certain part of the tweet documentation. MQTT events map to broker status: `connect`→connected, `reconnect`→connecting, `close`→disconnected, `error`→error.
- **Calls**: `sendCall` opens `net.createConnection({host: ip, port})`, writes the JSON and accumulates `data` until `end`/`close`, then runs `readReply`. A 15 s socket timeout gives `Unsuccessful: timeout waiting for X`. With no IP known it returns `Unsuccessful` immediately.
- `readReply`: `status` is `Successful` only if `Status === "Successful"`. `result` is `null` for `"No Output"`, otherwise `Number(...)`. The orchestrator needs a real number to compare.
- **Per-thing serialisation**: `queues: Map<thing, Promise>`. Each call chains onto the previous promise for that thing, so calls to one thing never overlap, while different things run in parallel. (Lab 4's lesson: Atlas handles one connection at a time.)

### 10.2 `gateway-sim.js` (no hardware)

- Announces **two things** with exactly the tweet shapes Atlas uses (`thingTweets()` and `serviceApi()` rebuild the `API` string, `NULL`s included), 60 ms apart:
  - `AT_BoarAirSensor`: the 3 entities, 10 services and 6 relationships of the XML, **copied exactly**. The house rule: *the simulator never invents a relationship*.
  - `AT_BoarLamp`: another group's Pi. Entity `Lamp`, services `Lamp_On(level 0–100)`, `Lamp_Off`, `Read_Lamp_Level`, **no relationships** (its IoT-DDL isn't in this repo). It exists so cross-thing apps and relationships can be tested.
  - Both advertise IP `127.0.0.1`, port 6668.
- **Behaviour (copied from Lab 4)**: inputs are **clamped** rather than refused (`RANGES`: Fan_On 0–75, Set_Window 0–50, Read_Temp 0–1, Lamp_On 0–100). Actuator reads report the last commanded value. `Read_Temp` = `22 + 8·sin(t/30 s)` °C, converted to °F if the argument is 1. `Read_Humidity` = `50 + 20·sin(t/41 s)`. These slow sine swings make condition-based apps actually change branch. `Calibrate_Sensor` always returns 1.
- **Errors**: an unknown service on that thing returns `Unknown Service`. A wrong argument count returns `Incorrect service arguments`.
- **Timing**: each call waits for the previous one plus `CALL_GAP_MS` (300 ms), then 300 ms more.

---

## 11. Server: apps, relationships, runtime

### 11.1 `server/apps.js`: the Atlas working directory (`AppStore`)

**`.iot` format** (JSON; the source text is canonical, the layout sits beside it):
```json
{ "format": "atlas-iot-app", "version": 1, "name": "AirControl",
  "created": "…", "modified": "…",
  "source": "RUN  AT_BoarAirSensor.Calibrate_Sensor()\nTHEN …",
  "layout": { "start": {"x":-136,"y":102}, "c:3o0ev0#0": {"x":60,"y":135} } }
```
The layout is keyed by **AST id** (see §12.3). Older files may still contain `"finalized": true`, which is ignored (finalize was dropped, conflict C4).

**Naming rules** (`nameError`, `fileNameFor`, `appNameOf`). These are graded rules, so know them:

| Typed | Result |
|---|---|
| `x` | `x.iot` |
| `x.ext` | `x.ext.iot` (R2 literally) |
| `x.iot` | `x.iot` (never doubled) |
| empty, > 32 chars, contains `/` or `\` | refused |
| base not matching `^[A-Za-z0-9_-]+(\.[A-Za-z0-9_-]+)?$` | refused: "may only use letters, digits, - and _" |

Other functions: `list()` returns `.iot` files only, sorted newest-first by mtime; an unreadable JSON file is still listed. `read()`. `write()` keeps the original `created` and overwrites silently, so repeated saves are fine. `remove()`. `resolve()` applies `basename()` and requires the `.iot` extension, so a path can never escape the directory even if it slipped past `nameError`.

### 11.2 `server/relationships.js`: user relationships (`RelationshipStore`)

File `relationships.json` in the working directory: `{ "format": "atlas-relationships", "version": 1, "relationships": [...] }`. `load()` treats a missing **or corrupt** file as an empty list, and filters out malformed records (`isWellFormed`). `list()` returns `structuredClone` copies.

**`buildRelationship(input, state)` validation, in order** (each message is shown verbatim in the dialog):
1. non-empty name, ≤ 40 characters
2. type ∈ `USER_TYPES` = `["order", "condition"]`
3. both ends given (`thing` + `service`)
4. not the same service twice (same thing and same service)
5. both ends actually offered by a discovered thing. The **entity is taken from the space, not from the browser**
6. for `condition`: the first service has a non-void output ("a condition relationship needs a first service that returns a value")
7. not a duplicate (same type, same two endpoints) of **any** known relationship, advertised ones included

Stored record: `{ id: "user:<randomUUID>", name, type, description, first: {thing, entity, service}, second: {…} }`.
`remove(id, state)`: advertised → `403` ("…was advertised by X and cannot be deleted"), unknown → `404`, otherwise splice, save, `204`.

### 11.3 `server/runtime.js`: the run life cycle (`RunManager`)

- `activate({file, name, source})`: refuses if the same file already has an `Active` run. Creates `{id: "r1"…, startedAt, status: "Active", signal: {stopped: false}}`, notifies, then calls `execute()` **without awaiting it**, so activation does not block.
- `execute()`: `runProgram(buildAst(source), {invoke, log, signal})`. The outcome `stopped` gives `Stopped`, anything else gives `Completed`. An exception gives `Stopped` plus an error log line. Sets `endedAt`.
- `stop(runId)`: errors if unknown or not `Active` (a **Completed app cannot be stopped**, R4). Otherwise sets `signal.stopped = true` and logs "stop requested".
- `invoke(call)`: **routing**. It finds the things that offer `call.name`. If the call is qualified it picks that thing, otherwise the **first** candidate, logging an ambiguity note if there are several. It passes `thing.ip`/`thing.port` from the `Identity_Language` tweet to `gateway.call`. If no thing matches: `Unsuccessful` with a readable reason.
- Statuses: **`Active` / `Stopped` / `Completed`**. Every log line is broadcast as `{runId, app, kind, depth, text}`.

---

## 12. The composition language (`lang/`)

### 12.1 Syntax (`lang/iot.grammar`, Lezer)

```
Program   := statement*
statement := RunStatement | IfStatement | LoopStatement | CallStatement
RunStatement  := RUN Call (THEN Call)*                          // Type 1 (single) / Type 2 (chain)
IfStatement   := IF Condition DO Block (ELSE Block)? END         // Type 3
LoopStatement := LOOP EVERY Duration Block END                   // added (D11) so STOP is demonstrable
CallStatement := Call                                            // a bare call (normal inside branches)
Condition := Predicate (LogicOp Predicate)*      Predicate := Call CompareOp Number
Call      := (ThingName ".")? ServiceName "(" (Argument ("," Argument)*)? ")"
Argument  := Number | Alias                      LogicOp := AND | OR
tokens: identifier [A-Za-z_][A-Za-z0-9_]*, Number -?digits, Duration digits"s", CompareOp >= <= == != > <, // comments
```

- **Keywords** are `@specialize`d identifiers, accepted in **all upper or all lower case only** (`RUN`/`run`, not `Run`). They are reserved, so no service can be called `RUN`.
- **Line breaks carry no meaning**. Statements are self-delimiting (a call ends at `)`, an IF at `END`).
- **No parentheses in conditions.** `AND` binds tighter than `OR`. To group, nest an `IF`.
- A predicate always compares **a service call** against **an integer literal**.
- Arguments are integers (negatives allowed) or the aliases **`C`=0, `F`=1**.
- `@precedence { Duration, Number, identifier }` makes `5s` one Duration token rather than `5` + `s`.

Example (`apps/AirControl.iot`):
```
RUN  AT_BoarAirSensor.Calibrate_Sensor()
THEN AT_BoarAirSensor.Read_Temp(C)
IF   AT_BoarAirSensor.Read_Temp(C) > 25 AND AT_BoarAirSensor.Read_Humidity() > 40
DO
     AT_BoarAirSensor.Fan_On(30)
     AT_BoarAirSensor.Set_Window(40)
ELSE
     AT_BoarAirSensor.Fan_Off()
     AT_BoarAirSensor.Window_Close()
END
```
`apps/NightWatch.iot` is a `LOOP EVERY 10s` with an `IF Read_Temp(C) > 26 DO Fan_On(40) ELSE Fan_Off() END` inside. It stays Active until stopped.

### 12.2 AST (`lang/ast.js`)

`buildAst(text, tree?)` walks the Lezer tree once and **skips error nodes** (validation reports them separately).

```jsonc
{ "kind": "program", "body": [
  { "id": "ord:ab12cd#0", "kind": "order", "bare": false, "calls": [call, …], "from", "to" },
  { "id": "if:…#0", "kind": "condition",
    "test": { "kind": "or", "terms": [ { "kind": "and", "terms": [predicate, …] }, … ] },
    "consequent": [stmt…], "alternative": [stmt…] },
  { "id": "lp:…#0", "kind": "loop", "everySeconds": 10, "body": [stmt…] } ] }
predicate = { id: "p:…", kind: "predicate", call, op, value }
call      = { id: "c:…", kind: "call", thing: "AT_…"|null, name, args: [{kind:"number",value}|{kind:"alias",name}] }
```

- The condition is always stored as an **OR of ANDs** (two levels), so evaluation and precedence are trivial.
- `bare` records whether `RUN` was written, so re-printing what the user typed does not add a one-word diff.
- Helpers: `callSignature(call)` → `Thing.Service(a, b)`, `argumentText`, `walk(ast, visit)` (depth-first over statements, predicates and calls).

### 12.3 Stable ids: the heart of synchronisation

`id = <prefix>:<hash6>#<n>`
- prefixes: `ord` (order statement), `if`, `lp` (loop), `p` (predicate), `c` (call)
- `hash6` = FNV-1a of the node's **own** text, base 36, last 6 characters. That is the call signature for calls and order statements, the condition text for IFs, and `every N` for loops.
- `#n` = how many identical ones came before, so two identical calls get different ids.

Because an IF's id comes from its condition text only, **editing a call inside an IF does not change the IF's id**. The block keeps its position and does not flash as "removed + added". The same id ties together: a Drawflow node (`data.astId`), the AST node, the printed lines, and the saved layout.

### 12.4 Printer (`lang/print.js`)

`printProgram(ast)` → `{ text, lines: Map<id, lineNumbers[]>, owners: Map<line, id> }`.
- A 5-column keyword gutter (`RUN  `, `THEN `, `IF   `, `DO   `, `ELSE `, `END`, `LOOP `), and branch bodies indented by +5.
- A single bare call prints without a keyword; a chain prints `RUN …` / `THEN …`.
- An IF prints its `IF`, `DO`, `ELSE` and `END` lines, **all owned by the IF's id**. Call lines are owned by the **call** ids (not the order statement's id), which matches the graph, where every call is its own block.
- Printing is **deterministic and idempotent** (print → parse → print gives the same text; this is tested).

### 12.5 Validation (`lang/validate.js`): deliberately thin

`validate(text, {catalogue})` → `{ errors, warnings, ast, tree }`. Three levels:

| Level | Source | Blocks Apply? | Examples |
|---|---|---|---|
| 1 Syntax | every Lezer **error node** | yes | zero-length node = "missing …" using the `EXPECTED` map (`END to close this IF`, `) to close the argument list`, `a comparison, like > 28`, …). Consumed text = Lab 4 wording: `"THEN" without a matching RUN`, `"ELSE"/"END" without a matching IF`, `unexpected "…"` |
| 2 Structure | AST checks | yes | nothing may follow a LOOP; an IF needs ≥1 command after DO; IF needs a condition; LOOP needs a body and > 0 seconds; a call needs a name; an alias that is not C/F |
| 3 Catalogue | the discovered services | **no, warnings only** | unknown service; `X does not offer Y`; wrong argument count; an ambiguous unqualified call ("qualify the call with a thing") |

Level 3 runs only if there are no errors, and never blocks, so an app can be written before its thing has been heard from. `catalogueFromSpace(state)` builds `Map<serviceName, [{thing, entity, inputs}]>`. Diagnostics carry `{from, to, line, column, message, severity}`. Zero-length ranges are widened to the rest of the line so there is always something to underline in red.

---

## 13. The orchestrator (`lang/run.js`)

Ported from Lab 4 `app.js`, **with its semantics kept exactly**. `runProgram(ast, {invoke, log, signal})` → `"completed" | "stopped"`.

| Rule | Implementation |
|---|---|
| Statements are independent (fault tolerance) | `runBlock` try/catches each statement and logs `error:`, then continues |
| Type 1: a single call | logs `[single call]`. Failure is just logged |
| Type 2: chain stops at the first unsuccessful call | `runOrder` logs `chain stopped: X was unsuccessful, N call(s) skipped`, or `chain completed` |
| Type 3: every predicate is a **real service call** | `evaluatePredicate` → `invokeCall`, then `compare(result, op, value)` |
| OR of ANDs with **short-circuit**, skipped predicates logged | `evaluateCondition`: inside an AND group, once one is false the rest are logged `skipped … (short-circuit)`; the first true group wins |
| If a predicate call fails (or returns no number), **neither branch runs** | `PredicateError` → `not evaluated: …, no branch run` |
| No ELSE and false | `skipped: no ELSE branch` |
| Timeout = failure | the gateway's 15 s timeout gives `Unsuccessful` |
| LOOP | `runLoop`: `pass 1, 2, …` until `signal.stopped`, sleeping `everySeconds` between passes |
| STOP | `signal.stopped` is checked before every statement and every call. `sleep()` polls every ≤ 100 ms. An in-flight call is **not** aborted: it finishes and nothing new starts |
| Arguments | `argumentValue`: numbers pass through, `C`→0, `F`→1, unknown alias→0 |

Log kinds: `info`, `call`, `ok`, `fail`, `error`, `other`, each with a `depth` used for indentation in the Output tab.

---

## 14. The graph model (`lang/graph.js`)

Pure functions, no DOM, so they are unit-tested in Node.

### 14.1 Node types and ports

```js
PORTS = { start: ["next"], service: ["next", "ok"], condition: ["then", "else", "next"], loop: ["body"] }
```

| Edge | Means | Prints as |
|---|---|---|
| `start.next` / `*.next` | the following statement (Type 1) | the next line |
| `service.ok` | run only if this call succeeded (Type 2) | `THEN` |
| `condition.then` / `.else` | the branches (Type 3) | `DO …` / `ELSE …` |
| `loop.body` | the loop body | indented inside `LOOP … END` |

**A loop has no `next`**: it runs until stopped, so nothing can follow it. Validation rejects a statement after a LOOP for the same reason. Every other block has a `next`, which is why the three types compose freely. Drawflow numbers ports `output_1`, `output_2`, … in the order of `PORTS`.

### 14.2 Functions

- `astToGraph(ast, {layout})` → a Drawflow export `{drawflow: {Home: {data: nodes}}}`. It creates one `start` node, one `service` node per call (chained by `ok` inside a `RUN…THEN`), one `condition` node per IF (its condition stored **as text** in `data.condition`), and one `loop` node per LOOP. Each node gets `data.astId`. The position comes from the saved layout, or from `autoPosition`: one row (95 px) down per block in print order and one column (270 px) right per nesting level, so the canvas reads like the text.
- `graphToAst(graph)` starts from `start.next` and follows chains (`readChain` / `readStatement`). **Unreachable blocks are dropped. Cycles cannot hang it** (a visited set). Only the **first** connection of each output port is followed. A single call at depth > 0 becomes `bare`. Empty argument fields are skipped.
- `tailOf(graph)` finds the free `next` port at the end of the program (skipping over THEN chains). A newly dropped block is attached there. It returns `null` if the program ends in a loop, in which case the new block is left unconnected.
- `layoutOf(graph)` → `{astId: {x, y}}`, saved into the `.iot`.
- `parseConditionText(text)` parses a condition block's text by wrapping it as `IF <text>\nDO\nEND` and reusing the same grammar. Any syntax error gives an empty condition.

---

## 15. The browser front end (`public/`)

Plain ES modules. Two classic scripts load first: `vendor/drawflow.min.js` (defines `window.Drawflow`) and `vendor/editor.bundle.js` (defines `window.AtlasEditor`). Then `main.js` loads as a module.

### 15.1 Layout (`index.html` + `style.css`)

```
┌ titlebar: "atlas ide" │ activate save stop delete │ metabar: 🌐 http://ip:7000 · BoarSmartSpace [broker: …] [discovery/N things] ┐
├ activity bar ┬ sidebar (Apps | Devices | Settings) ┬ tab bar (one tab per project, • = dirty)                                   ┤
│  ▣ apps      │                                     ├ graph pane (toolbar: + if, + loop, clear, zoom) │ text pane (Apply/Discard) │
│  ▤ devices   │                                     │ Drawflow canvas  (+ lock overlay when editing)  │ CodeMirror 6              │
│  ⚙ settings  │                                     ├ bottom panel: console | output | status   ⌫ ⌄                               ┤
└──────────────┴─────────────────────────────────────┴─────────────────────────────────────────────────────────────────────────┘
+ full-screen "scanning smart space…" overlay while discovery = scanning
```

VS Code-like dark theme only, built on CSS custom properties (`--bg`, `--added`, `--removed`, `--sidebar-w`, `--bottom-h`, `--text-pane-w`, …). Icons are an inline SVG sprite (`<use href="#i-apps">`), so nothing is fetched at runtime (D13).

### 15.2 Module by module

| File | Responsibility | Key functions |
|---|---|---|
| `main.js` | Bootstrap and chrome. Holds the latest `space` state and lets panels subscribe (`onSpaceChange`). SSE subscription (`connect`) or polling (`poll`). Meta bar pills, start-up overlay, project tab bar, menu actions, status rendering, Output log | `applyState`, `renderTabs`, `tabElement`, `activateActive` (**saves first** if new or dirty, since the server runs the file on disk), `stopActive`, `renderStatus`, `appendLog`, `openRequestedProject` |
| `project.js` | The **project model**: a list of open projects plus the active id. Each project owns `{id, name, path, ast, text, lines, owners, layout, editingText, editorState, dirty, savedText, console}`. Nothing global | `createProject`, `closeProject`, `setActiveProject`, `reprint`, `setAst`, `markDirty`, `markSaved`, `setConsole`, `onProjectsChange` |
| `editor-entry.js` | The **only file that knows CodeMirror**. Defines the language (Lezer parser + `styleTags` highlighting), the theme, the decoration field and `createEditor()`. Re-exports `lang/` functions. Bundled into `window.AtlasEditor` | `createEditor` → `{newState, state, useState, getText, setText, blankLines, dropLines, marks, clearChangeMarks, clearMarks, revealLine}` |
| `text-editor.js` | One CodeMirror view shared by all projects (each project keeps its own `EditorState`). The Apply/Discard protocol, diagnostics → console, graph → text highlighting | `showProject`, `syncFromModel`, `handleUserEdit`, `applyChanges`, `discardChanges`, `showDiagnostics`, `setCatalogue` |
| `graph-editor.js` | The Drawflow canvas: import/export, change events → model, drag-and-drop target, block HTML, the toolbar, relationship suggestions | `showProjectGraph`, `handleCanvasChange`/`flushCanvasChange`, `modelFromCanvas`, `addServiceBlock`, `addControlBlock`, `addBlock`, `dropRelationship`, `applySuggestion`, `nodeHtml` |
| `relationship-guidance.js` | DOM-free rules: which relationships can be built, which ones a dropped service suggests, and the text they generate (unit-tested) | `isActionable`, `suggestionsFor`, `callTextFor`, `conditionTextFor`, `suggestionLabel`. `GUIDED_TYPES = ["order","condition"]`, `PLACEHOLDER_VALUE = 0` |
| `panel-devices.js` | Devices panel: thing → (sensors, actuators) → entity → service rows (alphabetical, draggable, `act`/`rep` badges, tooltips with inputs and output). One space-wide **relationships** group with badges, a delete button for user rows, and the **add relationship `<dialog>`** | `render`, `thingGroup`, `entityGroup`, `serviceRow`, `relationshipsGroup`, `relationshipRow`, `endpointLabel`, `openAddDialog`, `fillServiceMenu`, `submitDialog` |
| `panel-apps.js` | Apps panel. Click selects, **double-click opens** (UPLOAD APP). Run badges. Save and delete | `refresh`, `openApp` (re-focuses the tab if already open), `saveActive`, `deleteActive` |
| `bottom-panel.js` | Three tabs: console (validation, the default), output (execution log), status (runs). A line is `{kind, where, what, reveal?, action?}` | `render`, `append`, `showTab`, `setCount`, `onBottomClear` |
| `settings.js` | `localStorage["atlas-ide-settings"]`: `highlightChanges` (true), `highlightRedMs` (700), `highlightFadeMs` (2000) | `getSetting`, `setSetting` |
| `resizers.js` | Draggable and keyboard-resizable splitters for sidebar, bottom panel and text pane (CSS vars persisted in localStorage; double-click resets). Comments are in Italian | `makeResizer`, `initResizers` |
| `dev/probe.html` | Dev aid, not part of the IDE: loads the IDE in an iframe, runs a scripted scenario (`lock`, `remove`, `add`, `invalid`, `apply`, `tabs`, `relationshipDrag`, …) and dumps a report. Used with headless Chrome | – |

### 15.3 The menu actions

| Button | Does |
|---|---|
| `activate` | save if needed → `POST /api/apps/:file/activate` → switch to the Output tab |
| `save` | `saveActive()`. The first save prompts for a name (paths refused client-side, then `nameError` server-side); later saves overwrite silently |
| `stop` | find the active run of this project's file → `POST /api/runs/:id/stop`; otherwise "X is not running" in Status |
| `delete` | refuse if running; `confirm()`; `DELETE /api/apps/:file`; close the tab |

All four are disabled while no project is open. *New app* is the `+` in the Apps panel (creates `untitled-N`). *Clear* is in the graph toolbar.

---

## 16. Keeping the two editors in sync (the core mechanism)

> **One sentence:** the **AST is the single source of truth**. The graph and the text are two views of it, and **stable AST ids** say exactly which lines and blocks were added or removed, so no text diffing is needed.

### 16.1 Graph → text (live, on every canvas change)

1. Drawflow fires `nodeCreated`, `nodeRemoved`, `connectionCreated`, `connectionRemoved` or `nodeDataChanged`. The last one fires when you type in a block's `<input df-arg0>`, because Drawflow binds `df-*` inputs to `node.data`.
2. Unless `suspended` is set (our own import), `handleCanvasChange()` records `beforeLines = project.lines` and **collapses the burst** with `setTimeout(…, 0)`. Deleting one block also removes its connections, and without this the second event would cut the first one's highlight short.
3. `flushCanvasChange()` → `setAst(project, modelFromCanvas())` → re-print → compare the **id sets**: `addedIds` = in the new lines but not the old, `removedIds` = the reverse. Then `rememberLayout()`.
4. **`modelFromCanvas()` trick:** new blocks carry placeholder ids (`c:new12345#0`). The canvas is converted to an AST and printed, the printed text is **re-parsed** to get canonical text-derived ids, and the two printings are matched **line by line** (`owners` maps). Each Drawflow node's `data.astId` is then renamed in place. So the canvas learns the ids the text editor would compute, **without re-importing the canvas** and interrupting the drag.
5. `text-editor.syncFromModel(project, {beforeLines, addedIds, removedIds})`:
   - highlighting **off** → just `setText` (the resulting document is identical; the setting changes only the visuals, D10);
   - only additions → `setText`, mark the added lines **green** (`cm-added`), fade after `highlightFadeMs` (2 s);
   - removals → `blankLines(removed)` (empty them in place) + mark them **red** (`cm-removed`), then after `highlightRedMs` (700 ms) `dropLines` and `setText`, then green any additions.
6. Every programmatic change is dispatched with the `fromModel` **annotation**. The update listener in `editor-entry.js` only calls `onUserEdit` for transactions **without** it, so the graph never locks itself.

### 16.2 Text → graph (explicit, on Apply Changes)

1. The **first user keystroke** (`handleUserEdit`) sets `project.editingText = true` and marks the project dirty. `updateToolbar()` sets `#graph-pane[data-locked=true]`, and CSS applies **`pointer-events: none` + blur** plus the "locked while you edit the text" overlay. Apply and Discard become enabled. Live validation keeps running and fills the console.
2. **Apply Changes** → `validate(text, {catalogue})`:
   - **errors** → red wavy underline on each range (`cm-invalid`), one console line each (`line:col message`, clickable to jump there), hint "N errors — nothing applied", **graph stays locked**;
   - **ok** → `setAst(project, result.ast)`, `setText(project.text)` (printing normalises spacing), unlock, then `reloadFromModel` → `showProjectGraph` → `editor.import(astToGraph(ast, {layout}))`. Surviving ids keep their positions; new ones are auto-laid out.
3. **Discard** → `reprint` the current AST back into the editor, clear the marks, unlock.
4. Warnings (dotted yellow `cm-warned`) never block Apply.

### 16.3 Selection sync

Clicking a block → `atlas-reveal-lines` event → the text editor highlights (`cm-reveal`) and scrolls to `project.lines.get(astId)`. Moving the cursor to a line (while not editing) → `owners.get(line)` → `atlas-reveal-node` → that block gets the `.revealed` outline.

### 16.4 Independent projects (multi-root workspace)

Each project has its own AST, layout, console buffer, dirty and lock flags, and **its own CodeMirror `EditorState`** (text, selection, undo history). There is **one** EditorView and **one** Drawflow instance. Switching tabs (`renderTabs` → only when the active id actually changed) saves the outgoing `editorState`, swaps in the new one (`useState`, with a `swapping` flag so the swap is not taken for typing), re-imports the new project's graph and renders its console. `PLAN.md` and `README.md` talk about "one Drawflow module per project", but the code always uses Drawflow's single `Home` module and **rebuilds the canvas from the project's AST and layout** on every switch. The effect is the same, since no graph state is shared between projects.

---

## 17. Relationships, end to end

| | **Advertised** (`source: "iot-ddl"`) | **User-defined** (`source: "user"`) |
|---|---|---|
| Origin | `Relationship` tweets from the IoT-DDL | the Devices panel's **+** dialog |
| Scope | same entity, so always the same thing | **any two discovered services**, cross-entity and cross-thing |
| Types | whatever the XML says (`order`, `interfere`, …) | only `order` and `condition` |
| Storage | none: relearned on every scan | `relationships.json` in the working directory, so it survives restarts |
| Delete | **refused (403)** | allowed (trash icon + confirm) |
| Id | `iot-ddl\|thing\|name\|first\|second` | `user:<uuid>` |

**Neither kind executes anything.** Atlas never runs relationships, and `runtime.js` never reads them. A relationship only **guides** the developer while building an app. It is shown in the panel, it can be dragged onto the canvas, and it is offered as a suggestion.

**In the UI**: one *relationships* group for the whole space, because a cross-thing relationship belongs to no single device. Each row reads `Entity.Service → Entity.Service`, prefixed with the thing when the two ends are on different things, and carries a type badge and a source badge (`IoT-DDL` / `user`). Unresolved rows (a thing offline) are greyed out but **never dropped**. The **+** button is disabled during the scan.

**The dialog**: name; type radios (*order: B runs only if A completes successfully*; *condition: B runs depending on A's return value*); two `<select>`s listing **every discovered service in the space**, grouped by `<optgroup>` "Thing · Entity" and labelled with the output type (`→ int` / `→ void`); an optional description. The server's error message is shown inline and the dialog stays open. On success, the server broadcasts a new `state` and every open window redraws.

**Guidance in the editor** (`relationship-guidance.js` + `graph-editor.js`). Only **actionable** relationships are used: `resolved` **and** type `order`/`condition`. `interfere` is never built, because "these two conflict" is not a program.

1. **Drag a relationship row** onto the canvas: `dropRelationship` adds a block for the first service at the program's tail, then `applySuggestion`.
2. **Drop a service**: `suggestionsFor(relationships, {thing, service})` finds the relationships whose **first** end is that `(thing, service)` and shows a small popup ("Read_Temp relates to: [then run Fan_On]").

| Type | What is built |
|---|---|
| `order` | a block for B connected to A's **`ok`** port → `RUN A THEN B` |
| `condition` | A's **`next`** → a condition block with text `Thing.A(0…) > 0` (placeholder value 0), and B in its **`then`** branch → `RUN A` … `IF Thing.A(…) > 0 DO B END` |

Matching is on **(thing, service)**, not the bare name. Generated calls are **always qualified** `Thing.Service(...)`, so the cross-thing case stays unambiguous.

---

## 18. Walkthroughs: what happens when…

### 18.1 …the IDE starts (`npm start -- --simulate`)
`index.js` parses args → creates Space, sim gateway, AppStore, RelationshipStore, RunManager → `listen` → loads `relationships.json` → prints the address → `startScanning()` (discovery = scanning) → the simulator emits 27 tweets over ≈1.6 s → each `applyTweet` emits `change` → SSE `state` to the browser → the overlay shows "heard N things so far" → 3 s of silence → `ready` → overlay hidden, the Devices panel is filled, the **+** button is enabled.

### 18.2 …you build the Lab 4 Part II demo graphically
Click **+** in Apps (an `untitled-1` project) → drag `Read_Temp` from Devices onto the canvas (`addServiceBlock` attaches it at `tailOf` = `start.next`; argument fields default to `0`) → type `C` into its field (`nodeDataChanged`) → **+ if** adds a condition block (default text `Read_Temp(C) > 25`, edit it to qualify) → drag `Fan_On` and `Set_Window` and wire them from the IF's `then` port → each change prints new lines in **green** on the right.

### 18.3 …you edit the text instead
Type in CodeMirror → the graph blurs and locks → write `IF … DO … END` without the `END` → the console shows `missing END to close this IF` in red → fix it → **Apply Changes** → the graph is rebuilt, unlocked, positions kept for unchanged ids.

### 18.4 …you save
`save` → first time `prompt("Save app as…")` → `PUT /api/apps/AirControl` → the server turns it into `AirControl.iot` and writes `{format, version, name, created, modified, source, layout}` → broadcasts `apps` → the tab is renamed and its dirty dot clears. The next `save` overwrites silently.

### 18.5 …you activate and stop
`activate` → saves if dirty → `POST /api/apps/AirControl.iot/activate` → `RunManager.activate` → `runs` SSE (Status row: start time, `active`, a stop button) → `runProgram` → each call goes `invoke` → route to a thing → `gateway.call` (TCP or simulated) → each `log` SSE line appears in **Output**, indented by depth → a finite app ends **Completed** (stop button disabled). A LOOP app stays **Active** until `stop` sets `signal.stopped`, after which the loop exits at its next check and the status becomes **Stopped**.

### 18.6 …you delete
`delete` → refused client-side if the app is running → `confirm` → `DELETE /api/apps/X.iot` → file unlinked, `apps` broadcast, the tab is closed.

---

## 19. Tests

Run with `node --test "tests/*.test.js"`: **134 tests, all passing**, with no browser and no hardware.

| File | Covers |
|---|---|
| `lang.test.js` | the three app types and LOOP parse; comments, case and aliases; printer gutter, idempotency, bare vs RUN, line ownership; **id stability**; every syntax and structure error and its line; catalogue warnings never block |
| `graph.test.js` | AST → graph → AST round-trips (sequence, chain, condition, Lab 4 demo, nested + loop); ports; ast ids on blocks; layout; `tailOf` cases; unreachable blocks dropped; cycles don't hang; no start block; condition text parsing |
| `run.test.js` | orchestrator semantics: order, chain stop, fault tolerance, branches, short-circuit AND/OR, failing predicate → no branch, every operator, C/F, negatives, loop until stopped, stop mid-chain, completion, **two things in one app** |
| `space.test.js` | `parseServiceApi`; identity merge; no duplicates; entity created by a service tweet; alphabetical order; sensor/actuator role; unknown tweets ignored; scan ends when quiet / at the max; the simulator announces both things, keeps Lab 4 behaviour and refuses bad calls |
| `apps.test.js` | every naming and extension rule; path escape; write/read; overwrite keeps `created`; old `finalized` files; scan order; bad JSON listed; delete |
| `relationships.test.js` | late Service tweet resolves later; stable advertised id; unresolved kept; every validation rule; cross-thing allowed; persistence across instances; delete rules (403/404); corrupt file; guidance rules; qualified generated text; HTTP POST/DELETE/400/403/404 through a spawned server |
| `server.test.js` | spawns the server on port 7123: serves the shell with all layout ids, state has the address, no path traversal, unknown → 404 |

---

## 20. Design decisions and documented conflicts

**Decisions (PLAN §12)**
- **D1** Node server + plain-JS browser IDE; no hardware available, so development runs on the simulator.
- **D2** `update` = SAVE (later renamed `save`); UPLOAD = double-click; new = `+`; clear = graph toolbar.
- **D3** Bottom panel = Console (default) / Output / Status.
- **D4** One space-level relationships group plus editor suggestions; user relationships persisted (revised 27 Nov).
- **D5** Device = Atlas thing; entities split sensor/actuator by service `Type`.
- **D6** Lezer grammar; Lab 4 orchestrator ported, its regex parser retired.
- **D7** Lab 4 language kept, only change: Thing ID qualifier (unqualified allowed when unambiguous).
- **D8** Meta bar shows the LAN URL, the space id and the broker/discovery pills.
- **D9** Settings behind a gear (see the limitation in §21).
- **D10** Red ≈700 ms then delete; green ≈2 s fade; the switch never changes the resulting document.
- **D11** `LOOP EVERY Ns … END` added so `stop` can be demonstrated.
- **D12** One Pi, but multi-thing design (qualified, routed, serialised per thing).
- **D13** Dark theme only, inline SVG icons, no skins.

**Conflicts (`directive_conflicts.md`; the prompt always wins)**
- **C1** Four discovery tabs (guidelines) vs two activity icons (prompt) → Things, Services and Relationships are folded into the Devices panel.
- **C2** SAVE/UPLOAD/ACTIVATE/STOP/DELETE/Clear/Finalize vs a fixed 4-item menu → mapping above; finalize dropped.
- **C3** Status panel vs console at the bottom → one panel with three tabs.
- **C4** Finalize as the only way to publish → `save` publishes; the `finalized` flag is ignored.
- **C5** Text **or** GUI vs both in sync → both (the guidelines allow it).
- **C6** "No build step" vs CodeMirror 6 (ES modules only) → bundle committed, listed in PLAN as AGENTS.md allows.
- **C7** File names → content in `PLAN.md` / `directive_conflicts.md`, with pointers.
- **C8** "Relationships between services of the same thing" vs user-defined cross-thing ones → both, told apart by `source`.
- **C9** "Work on main" vs "work on a branch" → built on a branch, merged with a real merge commit.

---

## 21. Known limitations, gotchas and doc-vs-code differences

These are the things an examiner might probe, or that you should mention honestly.

1. **Never run against real hardware.** Tweet shapes come from reading the middleware source; `gateway-atlas.js` is untested on a live thing (CHECKPOINT, step 11 blocked).
2. **`npm test` fails on Node 22** (`node --test tests/` with a directory). Use `node --test "tests/*.test.js"`.
3. **Default broker mismatch**: the server defaults to `192.168.8.189:1883`, while the current IoT-DDL uses Hamachi `25.16.192.196`. Pass `--broker`.
4. **Reply description is lost on real Atlas failures**: Atlas puts the reason in `"Status Description"`, but `gateway-atlas.js` reads `parsed.Description`. A real `Unsuccessful` reply therefore shows an empty reason in the log, while the simulator's reasons do show.
5. **Delete-while-running is only checked in the browser.** `PLAN.md` says the server refuses it, but `DELETE /api/apps/:file` has no run check. A direct API call would delete the file of a running app (the run itself continues from memory).
6. **The server does not validate before running.** `runtime.execute` calls `buildAst(source)`, which silently skips syntax-error nodes, so a hand-edited file with errors runs whatever parses.
7. **Unqualified calls with several providers** run on the **first** candidate thing (logged as ambiguous) rather than being refused.
8. **The settings panel only has the highlight switch and the two timings** (in `localStorage`). The working directory and broker are set **only by command-line flags** (the panel says so). PLAN D9 / R4 "changeable working directory" is satisfied via `--apps`, not via the UI.
9. **No IoT-DDL range warnings.** PLAN §6.4 lists "argument outside the IoT-DDL range" as a level-3 warning, but `validate.js` does not implement it. Clamping happens on the thing and in the simulator.
10. **No green outline on nodes after Apply** (PLAN §7.2 mentions one); the graph is simply re-imported.
11. **The condition block holds its predicate as text**, so even in the graphical editor part of a condition is typed. The default text from `+ if` is unqualified (`Read_Temp(C) > 25`). A malformed condition text silently becomes an empty condition (and then fails validation: "IF needs a condition").
12. **The condition suggestion calls A twice**: A stays as its own statement (`RUN A`), and the IF calls A again as its predicate. That matches the language (a predicate is a real call), but it is worth knowing.
13. **No `condition` relationship in the XML**, so the advertised half shows only `order` and `interfere`. A condition relationship has to be user-defined.
14. **`relationships.json` writes are not atomic** (plain `writeFile`; the roadmap asked for atomic writes). A missing or corrupt file is tolerated on load.
15. **The simulator serialises calls globally** (one `busyUntil` for both things), whereas the real gateway serialises **per thing**.
16. **STOP does not abort an in-flight TCP call.** It prevents the next one; the current call can take up to 15 s to time out.
17. **Atlas reads a call with a single `read()`** of 4048 bytes and creates a new CppMicroServices framework and installs the bundle **on every call** (slow but simple). It supports only 0–2 int inputs and int/void outputs.
18. **Tweets are hand-concatenated JSON** in C++: a quote in a description would break them.
19. **The `AT_BoarLamp` thing is fictional** (simulation only), there to show cross-thing apps without a second Pi.
20. A comment in `gateway-atlas.js` says "several tweets in one payload, one per line", but the code parses the payload as a single JSON. That is fine because Atlas publishes one tweet per MQTT message.

---

## 22. Likely questions and answers

**Architecture**

- **Why not a pure browser app?** Browsers cannot open raw TCP sockets (calls go to TCP 6668), cannot scan or write files in a working directory, and an app run must survive closing the tab. So a local Node server does discovery, files and execution (D1).
- **Why SSE and not WebSockets?** The live stream is one-way (server → browser). SSE is built into browsers (`EventSource`) and needs no server dependency. Browser → server uses plain `fetch`.
- **What is shared between browser and server?** The `lang/` modules: parse, AST, print and validate in both; `run.js` only on the server. The browser gets them through `editor.bundle.js` (`window.AtlasEditor`). The server imports them directly as ES modules.
- **How many runtime dependencies?** One: `mqtt`. Everything else is a dev/build dependency or vendored.

**Discovery**

- **How does the IDE discover things?** It subscribes to `/Atlas/#` on the broker. Atlas republishes its identity, language, entity, service and relationship tweets every 10 s. `space.applyTweet` merges each one by its keys.
- **How do you avoid duplicates?** Merge keys: thing id; entity id; service name within an entity; relationship name+first+second. A repeat replaces the entry.
- **How does the start-up freeze work?** `discovery = scanning` until 3 s with no new tweet (after at least one), or 15 s maximum. The browser shows a blocking overlay while scanning.
- **How do you know whether a service is a sensor or an actuator?** The service tweet's `Type`: `Report` = sensor, `Action` = actuator. An entity with any Action service is an actuator.
- **What if the Relationship tweet arrives before the Service tweet?** The entity is resolved when the state is serialised, so it fills in as soon as the service is heard. Until then it is `resolved: false`.

**Language and validation**

- **What are the three app types?** Type 1 is a sequence of statements. Type 2 is `RUN A THEN B` (B only if A succeeded). Type 3 is `IF A(...) op N DO … ELSE … END` (on A's return value). They compose sequentially, IFs nest, and LOOP is an extra.
- **Operator precedence?** AND binds tighter than OR, with no parentheses. Stored as an OR of ANDs. Nest IFs to group.
- **What does validation check?** Lexical/syntactic errors from Lezer error nodes, plus a few structural rules. Catalogue checks (unknown service, argument count, ambiguity) are warnings only. There is no type checker, as the prompt required.
- **Why are catalogue problems only warnings?** An app may be written before its thing is online. The IDE must never refuse to open a file.
- **How is "unmatched delimiter" detected?** Lezer produces a zero-length error node where a construct ends early. The parent node type (IfStatement, ArgList, …) is mapped to a message such as "missing END to close this IF".

**Editors and sync**

- **How do the two editors stay in sync?** Through one AST per project with stable ids. Graph changes → `graphToAst` → print → compare id sets → green/red lines. Text changes → validate → AST → `astToGraph` → `import`, only on Apply.
- **Why not diff the text?** Ids say exactly which nodes appeared or disappeared, so the highlighting is exact with no heuristics.
- **How are ids made stable?** A hash of the node's own text plus an occurrence counter. An IF's id depends only on its condition, so editing inside it keeps its id.
- **How does the text editor know a change is the user's?** Programmatic transactions carry a `fromModel` annotation. Only un-annotated document changes count as typing and lock the graph.
- **How is the graph locked?** `#graph-pane[data-locked=true]` → CSS `pointer-events: none` + blur + an overlay, until Apply succeeds or Discard is pressed.
- **What does turning highlighting off change?** Only the visuals. Removals become immediate. The resulting document is identical (D10).
- **How are projects independent?** Each project object holds its own AST, layout, CodeMirror `EditorState`, console, and dirty/lock flags. Switching tabs swaps all of it.
- **How does a new block get its id without re-importing the canvas?** `modelFromCanvas` prints the canvas, re-parses that text to get canonical ids, matches the two printings line by line, and renames `data.astId` in place.

**App manager and runtime**

- **What's in a `.iot` file?** JSON: format, version, name, created, modified, the **source text** (canonical) and the **layout** keyed by AST id.
- **Naming rules?** Letters, digits, `-` and `_`, at most 32 characters, optionally one `.ext`. `.iot` is always appended once; paths are refused.
- **What happens on activate?** The app is saved if needed, the server reads the file, creates run `rN` (Active), runs the orchestrator asynchronously, and streams log lines and status over SSE.
- **Completed vs Stopped?** A run that ends by itself is Completed and cannot be stopped. A run ended by STOP (or by an error) is Stopped. A LOOP app runs until stopped.
- **How does STOP clean up?** It sets `signal.stopped`. The orchestrator checks it before every statement and call, and the loop's sleep wakes within 100 ms. Nothing new is sent.
- **Why is the status panel empty after a restart?** Runs live only in server memory, as R4 asks.
- **How are calls routed to the right thing?** The qualifier (`AT_BoarAirSensor.`) chooses the thing. An unqualified call goes to the first thing that offers it. The IP and port come from that thing's `Identity_Language` tweet.
- **Why serialise calls per thing?** The Atlas TCP server handles one connection at a time (the Lab 4 lesson). Different things can still be called in parallel.

**Atlas / hardware**

- **How does Atlas turn the XML into a callable service?** `Service_Formula` (C code in the XML) → AutoGen `service.def` → template `service.tpl` → `main.cpp` → `lib<Name>.so` CppMicroServices bundle. At call time the bundle is installed, the signature is checked with `typeid` (`Validate`), and it is run through a `va_list` (`Evaluate`).
- **Why do the formulas assign the output variable instead of returning?** The generator skips any line containing `return`. The template returns the address of the output member.
- **How do the Read services know actuator state?** `Fan_On` / `Set_Window` write the value to `/tmp/fan_speed` / `/tmp/window_angle`, and the read services read those files.
- **What frequency is the PWM?** 19.2 MHz / 192 / 2000 = 50 Hz (suitable for the servo).
- **What does a failed DHT11 read return?** −1 with Status Successful (after 3 tries).

**Relationships**

- **Do relationships execute anything?** No. Neither Atlas nor the IDE executes them. They only guide the developer when building an app.
- **Advertised vs user?** Advertised ones come from tweets, are same-entity, read-only and not stored. User ones are order/condition only, can link any two services across things, are stored in `relationships.json` and can be deleted.
- **Validation rules for a new relationship?** Name (≤40), type order/condition, both ends, not the same service, both ends exist, condition needs a first service with an output, no duplicates.
- **Why is `interfere` not buildable?** It says two services conflict; that is not a program shape. It is shown but never suggested or draggable.
- **Why always generate qualified calls?** With two things offering the same service name, a bare call would be ambiguous. Qualified calls also match how service blocks print.

---

## 23. Glossary

| Term | Meaning |
|---|---|
| **Atlas** | IoT thing middleware (C++) running on each Raspberry Pi |
| **Thing** | one Atlas device (e.g. `AT_BoarAirSensor`); id = ATID |
| **Entity** | a component of a thing (sensor/actuator), e.g. `Fan` |
| **Service** | a callable function of an entity, e.g. `Fan_On(speed)` |
| **Smart space / VSS** | the set of things sharing a space id (`BoarSmartSpace`); id = SSID |
| **IoT-DDL** | IoT Device Description Language: the XML describing a thing (`Atlas_IoTDDL.xml`) |
| **Tweet** | a JSON announcement from a thing (5 discovery types + call/reply) |
| **Relationship** | a declared link between two services: `order`, `condition` (supported), `interfere` (shown only) |
| **Working directory** | the folder holding `.iot` apps and `relationships.json` (`--apps`, default `apps/`) |
| **Run** | one activation of an app (`rN`), status Active / Stopped / Completed |
| **Gateway** | the server-side adapter for discovery and calls: `atlas` (MQTT + TCP) or `simulator` |
| **AST id** | `prefix:hash#n`, the stable key tying text lines, graph blocks and layout together |
| **Lezer** | CodeMirror's incremental parser generator; `iot.grammar` is compiled by it |
| **Drawflow** | the node-graph editor library used for the canvas |
| **SSE** | Server-Sent Events: the one-way live stream `/api/events` |
| **Bundle** | a CppMicroServices plugin (`lib<Service>.so`) generated per service |
